In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (StructType, StructField, StringType, DecimalType, IntegerType, DateType)
from pyspark.sql import functions as F
from pyspark.sql.functions import round

print("1. Antes de criar SparkSession")
spark = SparkSession.builder.appName("beverage-sales").getOrCreate()
print("2. SparkSession criada")

print("3. Antes de ler CSV")
schema = StructType([
    StructField("Order_ID", StringType(), True),
    StructField("Customer_ID", StringType(), True),
    StructField("Customer_Type", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Unit_Price", DecimalType(10, 2), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DecimalType(5, 2), True),
    StructField("Total_Price", DecimalType(12, 2), True),
    StructField("Region", StringType(), True),
    StructField("Order_Date", DateType(), True)
])

df_raw = spark.read \
    .format("csv") \
    .option("header", True) \
    .option("delimiter", ",") \
    .option("mode", "FAILFAST") \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(schema) \
    .load("C:/Users/m24810/Projects/beverage-sales/data/raw/synthetic_beverage_sales_data.csv")

#.load("C:/Users/m24810/Projects/beverage-sales/data/raw/synthetic_beverage_sales_data.csv")
#.load("/home/rgerc/Projects/beverage-sales/data/raw/synthetic_beverage_sales_data.csv")
print("4. CSV lido")

print("5. Antes de count")
print(df_raw.count())
print("6. Count terminado")


#------------------------Data Cleaning------------------------#

print("Verify if there are any null values in the dataset:")
df_raw.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]).show()

print("Check for duplicate Order_IDs (it could make sense to have multiple entries for the same order):")
df_raw.groupBy("Order_ID").count().orderBy(F.desc("count")).show()

print("Check if Total_Price is correctly calculated:")
df_check = df_raw.withColumn(
    "Expected_Total",
    F.round(
        F.col("Unit_Price") * F.col("Quantity") * (1 - F.col("Discount")),
        2
    )
)

print("Show rows where Total_Price does not match the expected calculation:")
df_check.filter(
    F.abs(F.col("Expected_Total") - F.col("Total_Price")) > 0.01
).show()

print("Check for negative or zero values in Quantity and Unit_Price, and invalid Discount values:")
df_raw.filter(F.col("Quantity") <= 0).show()

df_raw.filter(F.col("Unit_Price") <= 0).show()

df_raw.filter((F.col("Discount") < 0) | (F.col("Discount") > 1)).show()

print("Some columns need to be derived for easier analysis:")
df_clean = df_raw \
    .withColumn("Year", F.year("Order_Date")) \
    .withColumn("Month", F.month("Order_Date")) \
    .withColumn("Gross_Sales", F.col("Unit_Price") * F.col("Quantity")) \
    .withColumn("Discount_Value", F.col("Gross_Sales") * F.col("Discount"))

df_clean.show(5)


1. Antes de criar SparkSession
2. SparkSession criada
3. Antes de ler CSV


Py4JJavaError: An error occurred while calling o36.load.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2078)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2122)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:1020)
	at org.apache.spark.util.HadoopFSUtils$.listLeafFiles(HadoopFSUtils.scala:218)
	at org.apache.spark.util.HadoopFSUtils$.$anonfun$parallelListLeafFilesInternal$1(HadoopFSUtils.scala:132)
	at scala.collection.immutable.List.map(List

In [6]:
df_clean.select("Category").distinct().show()

+-------------------+
|           Category|
+-------------------+
|              Water|
|Alcoholic Beverages|
|        Soft Drinks|
|             Juices|
+-------------------+



In [11]:
df_clean.select("Product").distinct().count()

47

In [7]:
df_clean.select("Customer_Type").distinct().show()

+-------------+
|Customer_Type|
+-------------+
|          B2C|
|          B2B|
+-------------+



In [8]:
df_clean.select("Region").distinct().show()

+--------------------+
|              Region|
+--------------------+
|      Sachsen-Anhalt|
|       Niedersachsen|
|         Brandenburg|
|              Berlin|
|              Bayern|
|             Sachsen|
|             Hamburg|
|              Bremen|
| Nordrhein-Westfalen|
|           Thüringen|
|              Hessen|
|   Baden-Württemberg|
|Mecklenburg-Vorpo...|
|            Saarland|
|  Schleswig-Holstein|
|     Rheinland-Pfalz|
+--------------------+

